## Process Data Enough to join into full Dataset

In [2]:
import pandas as pd

weather_df = pd.read_csv('../data/interim/weather_data.csv')
trappers_loop_df = pd.read_csv('../data/interim/trappers_loop_data.csv')

print(weather_df.head())
print(trappers_loop_df.head())

             ID = SBBWK  TMP ° F  DWP °F  RELH % QFLG   ALTI in  SOLR W/m*m  \
0  12-31-2016 23:45 MST     23.1    13.3    66.0    OK    29.58         0.0   
1  12-31-2016 23:30 MST     22.6    13.2    67.0    OK    29.59         0.0   
2  12-31-2016 23:15 MST     22.0    11.8    64.0    OK    29.59         0.0   
3  12-31-2016 23:00 MST     21.9    10.8    62.0    OK    29.60         0.0   
4  12-31-2016 22:45 MST     22.4    10.4    59.0    OK    29.60         0.0   

   VOLT volt  SKNT mph  GUST mph  DRCT °  SNOW in  P01I in  SINT in  
0      13.95       6.0      18.2   135.0    52.09      NaN     0.11  
1      13.95       4.6      13.4   139.0    52.14      NaN     0.10  
2      13.96       0.3       9.7    61.0    52.08      NaN     0.05  
3      13.95       2.0      10.2   231.0    52.04      0.0     0.07  
4      13.95       7.2       9.5   346.0    52.12      NaN     0.07  
   Unnamed: 0            Date_Hour    LANE  Count
0           0  2015-01-01 00:00:00  NLane1   96.0
1    

## Make weather Data hourly to match trappers loop

In [10]:

# Strip extra spaces and timezone
weather_df['timestamp'] = weather_df['ID = SBBWK'].str.replace(r'\s[A-Z]{2,3}$', '', regex=True)

# Convert to datetime
weather_df['timestamp'] = pd.to_datetime(weather_df['timestamp'], format="%m-%d-%Y %H:%M")

# Keep only rows on the hour
weather_hourly_df = weather_df[weather_df['timestamp'].dt.minute == 0]

print(weather_hourly_df.head())

              ID = SBBWK  TMP ° F  DWP °F  RELH % QFLG   ALTI in  SOLR W/m*m  \
3   12-31-2016 23:00 MST     21.9    10.8    62.0    OK    29.60         0.0   
7   12-31-2016 22:00 MST     21.8    10.2    60.0    OK    29.61         0.0   
11  12-31-2016 21:00 MST     20.2     6.0    53.0    OK    29.62         0.0   
15  12-31-2016 20:00 MST     20.6    10.3    64.0    OK    29.63         0.0   
19  12-31-2016 19:00 MST     19.6    10.0    66.0    OK    29.64         0.0   

    VOLT volt  SKNT mph  GUST mph  DRCT °  SNOW in  P01I in  SINT in  \
3       13.95       2.0      10.2   231.0    52.04      0.0     0.07   
7       13.99       3.2       8.4    15.0    52.13      0.0     0.02   
11      13.99       0.0       7.4     NaN    52.07      0.0     0.01   
15      13.99       0.9       6.8    93.0    52.03      0.0     0.00   
19      13.99       0.0       7.8     NaN    52.02      0.0     0.00   

             timestamp  
3  2016-12-31 23:00:00  
7  2016-12-31 22:00:00  
11 2016-12-

## Shift Trappers loop rows to have a column per hour not separated by p and n lanes

In [7]:
# Convert timestamp to datetime
trappers_loop_df['Date_Hour'] = pd.to_datetime(trappers_loop_df['Date_Hour'])

traffic_hourly_df = trappers_loop_df.pivot_table(
    index='Date_Hour',       # row index is the hour
    columns='LANE',          # column names from lane
    values='Count',          # values are counts
    aggfunc='first'          # only one count per lane per hour
).reset_index()

print(traffic_hourly_df.head())

LANE           Date_Hour  NLane1  PLane1
0    2015-01-01 00:00:00    96.0    85.0
1    2015-01-01 01:00:00    42.0    53.0
2    2015-01-01 02:00:00    10.0    30.0
3    2015-01-01 03:00:00     5.0     2.0
4    2015-01-01 04:00:00     8.0     5.0


## Join datasets

In [12]:

weather_hourly_df['timestamp'] = pd.to_datetime(weather_hourly_df['timestamp'])
traffic_hourly_df['Date_Hour'] = pd.to_datetime(traffic_hourly_df['Date_Hour'])

# Merge on timestamp using inner join
joined_df = pd.merge(
    traffic_hourly_df,
    weather_hourly_df,
    left_on='Date_Hour',
    right_on='timestamp',
    how='inner'  # keeps only rows where both traffic and weather exist
)

# Drop duplicate timestamp column from weather
joined_df = joined_df.drop(columns=['timestamp'])


print(joined_df.head())

# Save the joined dataset
joined_df.to_csv('../data/interim/joined_data.csv', index=False)



C:\Users\danir\AppData\Local\Temp\ipykernel_58772\2203714777.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  weather_hourly_df['timestamp'] = pd.to_datetime(weather_hourly_df['timestamp'])


            Date_Hour  NLane1  PLane1            ID = SBBWK  TMP ° F  DWP °F  \
0 2016-11-14 21:00:00    49.0   115.0  11-14-2016 21:00 MST     47.5    19.2   
1 2016-11-14 22:00:00    25.0    80.0  11-14-2016 22:00 MST     46.3    17.7   
2 2016-11-14 23:00:00     8.0    32.0  11-14-2016 23:00 MST     46.9    15.6   
3 2016-11-15 00:00:00     9.0    17.0   11-15-2016 0:00 MST     46.7    15.7   
4 2016-11-15 01:00:00     6.0     9.0   11-15-2016 1:00 MST     46.6    12.8   

   RELH % QFLG   ALTI in  SOLR W/m*m  VOLT volt  SKNT mph  GUST mph  DRCT °  \
0    32.0   NaN    30.05         0.0      13.68       NaN       NaN     NaN   
1    32.0   NaN    30.04         0.0      13.68       NaN       NaN     NaN   
2    28.0   NaN    30.04         0.0      13.68       NaN       NaN     NaN   
3    28.0   NaN    30.03         0.0      13.68       NaN       NaN     NaN   
4    25.0   NaN    30.02         0.0      13.76       NaN       NaN     NaN   

   SNOW in  P01I in  SINT in  
0      NaN   